In [2]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [3]:
import json
# Helper functions
from anthropic.types import Message
def add_user_message(messages, message):
    user_message = {"role": "user", 
                    "content": message.content if isinstance(message, Message) else message}
    messages.append(user_message)



def add_assistant_message(messages, message):
    assistant_message = {"role": "assistant", 
                         "content": message.content if isinstance(message, Message) else message}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools = None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    # add a new parameter tools and pass in the schemas
    if tools:
        params["tools"] = tools

    message = client.messages.create(**params)
    return message

# have a helper function that would help you seperate the contents of the returned message
def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

# run tool is only going to run a single tool given the name of the tool and the input
def run_tool(tool_name, Tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**Tool_input)

    # add aditional tools below as elif 
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**Tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**Tool_input)

# helper function that will run the tools for claude
def run_tools(message):

    # extract every tool use block that is mentioned in the message into tool_request list
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]

    tool_result_blocks = []

    for  tool_request in tool_requests:
        # call tool function and pass the output into results
        # use try and except to catch eeors during tool use
        try:
            is_error = False
            result = run_tool(tool_request.name, tool_request.input)
        except Exception as e:
            is_error = True
            result = {"error": str(e)}
            print(f"Error running tool {tool_request.name}: {e}")

        tool_result_blocks.append(
            {
                "content": json.dumps(result),
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "is_error": is_error
            }
        )

    # return the tool results in a single list of tool_result blocks
    return tool_result_blocks
        


# need a loop that runs until stop reason is not tool_use
def run_conversation(messages):
    while True:
        response = chat(messages, tools = [
            get_current_datetime_schema,
            add_duration_to_datetime_schema,
            set_reminder_schema,
            ])

        add_assistant_message(messages, response)
        print(text_from_message(response))

        # if claude stoped answering since it is asking for a tool to be run, we need to run the tool and add the result to the messages
        if response.stop_reason != "tool_use":
            break
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
        

    return messages

In [4]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [5]:
from anthropic.types import ToolParam

# this is example of a tool that calls the current date and time used by the Ai
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("Pass in a non-empty format into get_current_datetime")
    return datetime.now().strftime(date_format)

# this is the schema of the tool, this json schema is used to better help the Ai to udnerstand the tool and how to use it
# tool param is sepcific to anthropic, it wraps the schema inside this object so Claude can better understand it
get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Get the current local date and time, formatted according to a strftime format string. Use this whenever you need to know the present date or time (e.g. 'what time is it', 'what's today's date', timestamping output). Returns a single formatted string.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime format string controlling how the datetime is rendered. Must be non-empty. Examples: '%Y-%m-%d %H:%M:%S' -> '2026-07-29 14:30:00', '%Y-%m-%d' -> '2026-07-29', '%B %d, %Y' -> 'July 29, 2026', '%I:%M %p' -> '02:30 PM'. Defaults to '%Y-%m-%d %H:%M:%S' if the user has no specific preference.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
})
    

In [6]:
messages = []
messages.append(
    {
        "role":"user",
        "content": "what is the exact time formatted as HH:MM:SS"

    }
)

response = client.messages.create(
    model = model,
    max_tokens  = 600,
    messages = messages,
    tools = [get_current_datetime_schema],

)

messages.append({
    "role":"assistant",
    "content": response.content
})

messages

[{'role': 'user', 'content': 'what is the exact time formatted as HH:MM:SS'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01FaXjQ1jCMWaEdKnVyZ35t4', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

In [7]:
result = get_current_datetime(**response.content[0].input)

In [8]:
messages.append({
    "role":"user",
    "content": [
        {
            "type":"tool_result",
            "tool_use_id": response.content[0].id,
            "content": result,
            "is_error": False,
        }
    ]
})

In [9]:
message = client.messages.create(
    model=model,
    max_tokens=600,
    messages = messages,
    tools = [get_current_datetime_schema]
)
message.content[0].text

'The exact time is **22:42:51** (10:42:51 PM in 12-hour format).'

In [35]:
def lol():
    messages = []
    add_user_message(messages, "Whats the current time in HH:MM format? and in SS format, and set a reminder in 5 minuets from now to say 'Time to take a break!'")
    run_conversation(messages)
    print(messages)
   

lol()

I'll get the current time and set a reminder for you.
Now I'll set the reminder for 5 minutes from now:
Error running tool add_duration_to_datetime: unconverted data remains: T19:12:19
Let me fix the format:
Now I'll set the reminder:
----
Setting the following reminder for 2026-07-31T19:17:19:
Time to take a break!
----
Perfect! Here's the information:

- **Current time in HH:MM format:** 19:12
- **Current time in SS format:** 19 seconds
- **Reminder set:** ✓ A reminder to "Time to take a break!" has been set for 07:17 PM (5 minutes from now)
[{'role': 'user', 'content': "Whats the current time in HH:MM format? and in SS format, and set a reminder in 5 minuets from now to say 'Time to take a break!'"}, {'role': 'assistant', 'content': [TextBlock(citations=None, text="I'll get the current time and set a reminder for you.", type='text'), ToolUseBlock(id='toolu_01AYyHhGimdmgXWT3qnBdHP1', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M'}, name='get_current_datetime', type